<a href="https://colab.research.google.com/github/Vivekshrotriya1/11_April_CapstoneProject/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Capstone Scenario 2: Automated Resume Screening System
# Scenario
A company receives 1000+ resumes daily and wants to automate candidate screening.

They need a system that:

Extracts candidate details

Structures the data

Tags each processed document with Student ID (who processed it)

 Your Task
Build a document processing system using Azure.

 Requirements
You MUST use:

Azure Document Intelligence

Azure Cognitive Services

 Functional Requirements
Upload resume (PDF/Image)

Extract:

Name

Skills

Email

Attach Student ID to output

Use:

API Key

Endpoint

 Input
Resume file

Student ID

 Expected Output

{
  "student_id": "BTECH2026_205",
  "name": "Rahul Sharma",
  "skills": ["Python", "Azure"],
  "email": "rahul@email.com"
}

 Constraints

Handle different resume formats

Ensure structured JSON output

 Evaluation Focus

Accuracy of extraction

API usage

Real-world applicability

In [14]:
import requests
import json
import re
from google.colab import userdata
API_KEY=userdata.get('PROJECT2')
ENDPOINT = "https://viv54321.cognitiveservices.azure.com/"

def extract_resume(file_path, student_id):

    url = f"{ENDPOINT}/formrecognizer/documentModels/prebuilt-layout:analyze?api-version=2023-07-31"

    headers = {
        "Content-Type": "application/octet-stream",
        "Ocp-Apim-Subscription-Key": API_KEY
    }

    with open(file_path, "rb") as f:
        response = requests.post(url, headers=headers, data=f)

    # Step 1: Get operation location
    operation_url = response.headers["Operation-Location"]

    # Step 2: Fetch result
    while True:
        result = requests.get(operation_url, headers=headers).json()
        if result["status"] == "succeeded":
            break

    # Step 3: Extract text
    text = ""
    for page in result["analyzeResult"]["pages"]:
        for line in page["lines"]:
            text += line["content"] + "\n"

    # Step 4: Simple extraction logic
    name = text.split("\n")[0]  # first line as name

    emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)
    email = emails[0] if emails else ""

    skills = []
    skill_keywords = ["Python", "Java", "Azure", "SQL", "ML"]

    for skill in skill_keywords:
        if skill.lower() in text.lower():
            skills.append(skill)

    # Step 5: Final JSON
    output = {
        "student_id": student_id,
        "name": name,
        "skills": skills,
        "email": email
    }

    return output


# Example
result = extract_resume("VIVEK_RESUME.pdf", "BTECH2026_226")
print(json.dumps(result, indent=2))

{
  "student_id": "BTECH2026_226",
  "name": "VIVEK SHROTRIYA",
  "skills": [
    "Python",
    "Java",
    "SQL",
    "ML"
  ],
  "email": "123@gmail.com"
}
